In [1]:
import sys, torch
sys.path.append("../src")
from model import DrowsinessDetector

model = DrowsinessDetector()
model.eval()

# Dummy batch: 2 samples, 16 frames each, 3 channels, 224x224
dummy = torch.randn(2, 16, 3, 224, 224)

with torch.no_grad():
    out = model(dummy)

print(f"Input shape  : {dummy.shape}")
print(f"Output shape : {out.shape}")   # must be torch.Size([2, 2])
print(f"Output dtype : {out.dtype}")   # must be torch.float32
print(f"Output values: {out}")          # raw logits — any real numbers
print("\nShape check PASSED" if out.shape == torch.Size([2, 2]) else "FAILED")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\mitta/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:04<00:00, 5.14MB/s]


Input shape  : torch.Size([2, 16, 3, 224, 224])
Output shape : torch.Size([2, 2])
Output dtype : torch.float32
Output values: tensor([[-0.0484,  0.3018],
        [-0.0450,  0.3033]])

Shape check PASSED


In [2]:
total, trainable = model.count_parameters()

# Phase 1 (CNN frozen): only LSTM + SE + classifier are trainable
# Expected: ~4-5M trainable out of ~9M total
# If trainable == total, CNN is NOT frozen — check __init__

print(f"\nFrozen fraction: {(total-trainable)/total*100:.1f}%")
print("Phase 1 training: CNN frozen, training LSTM + head only")

Total params     : 9,006,078
Trainable params : 4,998,530
Frozen params    : 4,007,548

Frozen fraction: 44.5%
Phase 1 training: CNN frozen, training LSTM + head only


In [3]:
print("Before unfreeze:")
model.count_parameters()

model.unfreeze_cnn(blocks=2)
print("\nAfter unfreeze_cnn(blocks=2) — Phase 2:")
model.count_parameters()

model.unfreeze_all_cnn()
print("\nAfter unfreeze_all_cnn() — Phase 3:")
model.count_parameters()

# Reset to frozen state for actual training start
for p in model.cnn.parameters():
    p.requires_grad = False
print("\nReset to frozen for training start.")

Before unfreeze:
Total params     : 9,006,078
Trainable params : 4,998,530
Frozen params    : 4,007,548

After unfreeze_cnn(blocks=2) — Phase 2:
Total params     : 9,006,078
Trainable params : 6,127,922
Frozen params    : 2,878,156

After unfreeze_all_cnn() — Phase 3:
Total params     : 9,006,078
Trainable params : 9,006,078
Frozen params    : 0

Reset to frozen for training start.


In [4]:
from dataset import get_dataloaders

loaders = get_dataloaders(
    data_dir="../data/processed",
    batch_size=4,
    num_workers=0
)

# Get one real batch of images
real_imgs, real_labels = next(iter(loaders["train"]))
# real_imgs shape: (4, 3, 224, 224) — single frames, not sequences
# We need to fake a sequence by repeating the batch along a new dim
seq_batch = real_imgs.unsqueeze(1).repeat(1, 16, 1, 1, 1)
# seq_batch shape: (4, 16, 3, 224, 224)

model.eval()
with torch.no_grad():
    logits = model(seq_batch)
    probs  = torch.softmax(logits, dim=1)

print(f"Real batch input : {seq_batch.shape}")
print(f"Logits           : {logits.shape}")   # (4, 2)
print(f"Probabilities    :\n{probs}")         # rows sum to 1.0
print(f"Predictions      : {probs.argmax(dim=1).tolist()}")
print(f"True labels      : {real_labels.tolist()}")
print("\nModel + DataLoader integration: PASSED")

  train: 84,044 images | open=41,085 closed=42,959
  val  : 21,056 images | open=10,456 closed=10,600
  test : 21,063 images | open=10,464 closed=10,599
Real batch input : torch.Size([4, 16, 3, 224, 224])
Logits           : torch.Size([4, 2])
Probabilities    :
tensor([[0.5249, 0.4751],
        [0.4393, 0.5607],
        [0.4126, 0.5874],
        [0.3750, 0.6250]])
Predictions      : [0, 1, 1, 1]
True labels      : [0, 0, 1, 0]

Model + DataLoader integration: PASSED
